# Diniz Ch.7 — Floating and fixed-point lattice RLS

Problem 17：16-tap system、λ=0.98。明確設定 16-bit data/coefficient 與 32-bit energy state；原 notebook 的 bc/bd 參數不存在於上傳的 class。

來源：`Diniz-Chap7-SignalPred.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Data and quantization formats

In [ ]:
from adsp.experiments import diniz
from adsp.quantization import FixedQuantizer

n_samples, n_trials, seed = 1500, 20, 2026
# Original n_trials=200. Wider energy states avoid overflow in accumulated lattice powers.
make_data = partial(data.system_identification, n_samples, h=diniz.FIXED_LATTICE_WEIGHTS, input_variance=0.7,
                    noise_var=0.0015)
models = {
  "Floating LRLS":
  partial(af.LatticeRLS, filter_order=15, lam=0.98, epsilon=0.1),
  "Fixed LRLS":
  partial(af.QuantizedLatticeRLS, filter_order=15, lam=0.98, epsilon=0.1, data_q=FixedQuantizer(16, 12),
          coef_q=FixedQuantizer(16, 13), state_q=FixedQuantizer(32, 12)),
}

## 3. Training and reflection-coefficient traces

In [ ]:
results = training.compare(models, make_data, n_trials=n_trials, seed=seed, trace_keys=("kappa_f", "kappa_b"))
plot.learning(results, noise_floor=0.0015, window=64, title="Physical MSE: floating versus fixed-point LRLS")
plot.show()
floating, fixed = (results[name].representative for name in results)
differences = {
  key: np.sum(np.abs(floating.extras[key] - fixed.extras[key])**2, axis=1)
  for key in ("kappa_f", "kappa_b")
}
plot.lines(differences, scale="log", window=21, ylabel="Squared coefficient difference",
           title="One-trial reflection-coefficient quantization error")
plot.show()
training.summary(results, tail=500, noise_floor=0.0015)